In [7]:
import pickle as pk
import pandas as pd
import numpy as np

import shap


In [2]:
df= pd.read_csv("synthetic_subsidy_cylinders2.csv")
with open('elmodel.pk', 'rb') as file:
    eligibity_model = pk.load(file)
df
with open("frmodel.pk",'rb') as file:
    fraud_model = pk.load(file)


In [4]:
#ID=14129593
#re = df.loc[df["ID"] == ID].drop(columns=["ID"])
#pr=eligibity_model.predict(re.drop(columns=[ "Eligible","Fraud"]))
#x=int(pr[0])
#print(x)
#pf=fraud_model.predict(re.drop(columns=["Fraud"]))
#x1=int(pf[0])
#explainer = shap.TreeExplainer(eligibity_model)
#shap_values = explainer.shap_values(df.loc[df["ID"] == ID].drop(columns=["ID", "Eligible","Fraud"]))
#print(df.columns.shape)
#column_names = df.columns.drop(["ID", "Eligible", "Fraud"])
#adf=pd.DataFrame(columns=column_names ,data=shap_values)
#print(adf,adf.min().idxmin(),adf["Salary"])

In [6]:
from flask import Flask ,jsonify
from flask_cors import CORS


app = Flask(__name__)
CORS(app)

@app.route("/")
def a():
    return "A"
@app.route('/EEml/<int:ID>/<string:_id>',methods=["GET","PUT","POST"])
def EEml(ID,_id):
    try:
        re = df.loc[df["ID"] == ID].drop(columns=["ID"])
        if re.empty:    
            return jsonify({"error":"notfound"}),404
        else:
            pr=eligibity_model.predict(re.drop(columns=[ "Eligible","Fraud"]))
            x=int(pr[0])
            pf=fraud_model.predict(re.drop(columns=["Fraud"]))
            x1=int(pf[0])
            if x1 == -1:
                x1=1
            else:
                x1=0
            if x==0:
                explainer = shap.TreeExplainer(eligibity_model)
                shap_values = explainer.shap_values(df.loc[df["ID"] == ID].drop(columns=["ID", "Eligible","Fraud"]))
                column_names = df.columns.drop(["ID", "Eligible", "Fraud"])
                eligire=pd.DataFrame(columns=column_names ,data=shap_values)
                return jsonify({"Eligibity":x,"Fraud":x1,"reson":eligire.max().idxmin()}),200
            else:
                return jsonify({"Eligibity":x,"Fraud":x1}),200       
    except SystemError:
        return jsonify({"error":"SystemError"}),500        
if __name__ == '__main__':
    app.run()

 * Tip: There are .env files present. Install python-dotenv to use them.


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [13/Apr/2026 02:05:56] "GET /EEml/15593616/69a88024c79be2c8ce08be8f HTTP/1.1" 200 -
